# Imports and config setup

In [4]:
import sys
from pathlib import Path
from time import perf_counter
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight


In [5]:
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [6]:
from walkforward import (
    FROZEN_FOLDS,
    build_walkforward_splits,
)

from model_evaluation import (
    evaluate_fold_scores,
    build_fold_metrics_table,
    summarize_temporal_stability,
)

In [7]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "training_data.parquet"

dev_df = pd.read_parquet(DATA_PATH)

dev_df["transaction_timestamp"] = pd.to_datetime(dev_df["transaction_timestamp"])

print(dev_df.shape)
print(dev_df["transaction_timestamp"].min())
print(dev_df["transaction_timestamp"].max())

(3731672, 20)
2022-09-01 00:00:00
2022-09-07 23:59:00


In [8]:
TEST_START = pd.Timestamp("2022-09-08")

assert (dev_df["transaction_timestamp"] < TEST_START).all(), (
    "September 8+ data detected!"
)

## the baseline feature set

In [9]:
NUMERIC_FEATURES = [
    "amount_received",
    "amount_paid",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "same_currency_flag",
    "same_bank_flag",
    "log_amount_received",
    "log_amount_paid",
]

CATEGORICAL_FEATURES = [
    "receiving_currency",
    "payment_currency",
    "payment_format",
]

FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

TARGET_COL = "is_laundering"
TIMESTAMP_COL = "transaction_timestamp"

In [10]:
missing = [
    col
    for col in FEATURE_COLUMNS + [TARGET_COL, TIMESTAMP_COL]
    if col not in dev_df.columns
]

missing

[]

# Verify our frozen folds

In [11]:
splits = build_walkforward_splits(dev_df)

In [12]:
for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    y_train = dev_df.iloc[train_idx][TARGET_COL]
    y_val = dev_df.iloc[val_idx][TARGET_COL]

    print(
        fold.name,
        "| train:",
        f"{len(train_idx):,}",
        f"fraud={y_train.sum():,}",
        "| val:",
        f"{len(val_idx):,}",
        f"fraud={y_val.sum():,}",
    )

fold_1 | train: 2,076,752 fraud=1,121 | val: 207,430 fraud=407
fold_2 | train: 2,284,182 fraud=1,528 | val: 482,650 fraud=471
fold_3 | train: 2,766,832 fraud=1,999 | val: 964,840 fraud=1,028


# Preprocessing function

In [13]:
def make_preprocessor():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                NUMERIC_FEATURES,
            ),
            (
                "categorical",
                categorical_pipeline,
                CATEGORICAL_FEATURES,
            ),
        ],
        sparse_threshold=1.0,
    )

# LightBGM

## LightGBM configuration

In [31]:
def make_lightgbm_pipeline():
    model = LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=8,
        min_child_samples=100,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    return Pipeline(
        steps=[
            ("preprocessor", make_preprocessor()),
            ("model", model),
        ]
    )

## Run Fold 1

In [32]:
fold = FROZEN_FOLDS[0]
train_idx, val_idx = splits[0]


In [33]:
X_train = dev_df.iloc[train_idx][FEATURE_COLUMNS]
y_train = dev_df.iloc[train_idx][TARGET_COL].to_numpy()

X_val = dev_df.iloc[val_idx][FEATURE_COLUMNS]
y_val = dev_df.iloc[val_idx][TARGET_COL].to_numpy()


In [34]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)

print("Train fraud:", y_train.sum())
print("Validation fraud:", y_val.sum())

Train: (2076752, 12)
Validation: (207430, 12)
Train fraud: 1121
Validation fraud: 407


## Calculate balanced sample weights

In [35]:
train_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train,
)

In [36]:
positive_weight = train_weights[y_train == 1][0]
negative_weight = train_weights[y_train == 0][0]

print("Positive weight:", positive_weight)
print("Negative weight:", negative_weight)
print(
    "Weight ratio:",
    positive_weight / negative_weight,
)

Positive weight: 926.2943800178413
Negative weight: 0.5002700383642372
Weight ratio: 1851.5887600356823


## Train Fold 1

In [37]:
pipeline = make_lightgbm_pipeline()

In [38]:
start = perf_counter()

In [39]:
pipeline.fit(
    X_train,
    y_train,
    model__sample_weight=train_weights,
)

fit_seconds = perf_counter() - start

print(f"Fold 1 training time: {fit_seconds:.2f} seconds")

Fold 1 training time: 24.61 seconds


## Predict Fold 1

In [40]:
start = perf_counter()

val_scores = pipeline.predict_proba(X_val)[:, 1]

predict_seconds = perf_counter() - start

print(f"Prediction time: {predict_seconds:.2f} seconds")

Prediction time: 0.80 seconds


In [41]:
pd.Series(val_scores).describe(
    percentiles=[
        0.5,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

count    207430.000000
mean          0.208021
std           0.255304
min           0.000026
50%           0.112209
75%           0.247783
90%           0.595883
95%           0.932957
99%           0.976659
max           0.994879
dtype: float64

## Evaluate Fold 1

In [42]:
fold_1_metrics = evaluate_fold_scores(
    fold_name="fold_1",
    y_true=y_val,
    y_score=val_scores,
    min_recall=0.80,
)

fold_1_metrics

{'fold': 'fold_1',
 'rows': 207430,
 'frauds': 407,
 'prevalence': 0.0019621076989827894,
 'average_precision': 0.06307894817449604,
 'pr_lift': 32.148565650702,
 'roc_auc': 0.9419330682209686,
 'recall_floor': 0.8,
 'diagnostic_cutoff': 0.85968196764517,
 'precision_at_recall_floor': 0.02277013340783684,
 'recall_at_recall_floor': 0.800982800982801,
 'false_positives_at_recall_floor': 13991,
 'true_positives_at_recall_floor': 326,
 'false_negatives_at_recall_floor': 81,
 'true_negatives_at_recall_floor': 193032,
 'alerts_at_recall_floor': 14317,
 'alert_rate_at_recall_floor': 0.06902087451188353}

In [43]:
print(f"Average Precision: {fold_1_metrics['average_precision']:.6f}")

print(f"PR lift: {fold_1_metrics['pr_lift']:.2f}x")

print(f"ROC-AUC: {fold_1_metrics['roc_auc']:.6f}")

print(f"Recall: {fold_1_metrics['recall_at_recall_floor']:.4%}")

print(f"Precision @ >=80% recall: {fold_1_metrics['precision_at_recall_floor']:.4%}")

print(
    f"False positives @ >=80% recall: "
    f"{fold_1_metrics['false_positives_at_recall_floor']:,}"
)

print(f"Alerts @ >=80% recall: {fold_1_metrics['alerts_at_recall_floor']:,}")

print(f"Alert rate @ >=80% recall: {fold_1_metrics['alert_rate_at_recall_floor']:.4%}")

Average Precision: 0.063079
PR lift: 32.15x
ROC-AUC: 0.941933
Recall: 80.0983%
Precision @ >=80% recall: 2.2770%
False positives @ >=80% recall: 13,991
Alerts @ >=80% recall: 14,317
Alert rate @ >=80% recall: 6.9021%


## run Folds 2 and 3

In [44]:
fold_results = [fold_1_metrics]
runtime_results = []

for fold_number in [1, 2]:
    fold = FROZEN_FOLDS[fold_number]
    train_idx, val_idx = splits[fold_number]

    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    # -----------------------------------------
    # Fold data
    # -----------------------------------------

    X_train = dev_df.iloc[train_idx][FEATURE_COLUMNS]
    y_train = dev_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = dev_df.iloc[val_idx][FEATURE_COLUMNS]
    y_val = dev_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    # -----------------------------------------
    # Training-only balanced weights
    # -----------------------------------------

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    # -----------------------------------------
    # Fresh pipeline for this fold
    # -----------------------------------------

    pipeline = make_lightgbm_pipeline()

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    # -----------------------------------------
    # Validation scores
    # -----------------------------------------

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    # -----------------------------------------
    # Common evaluator
    # -----------------------------------------

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    fold_results.append(metrics)

    runtime_results.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")

    print(f"PR lift   : {metrics['pr_lift']:.2f}x")

    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")

    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")

    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")

    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")

    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")

    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")

    print(f"Fit time  : {fit_seconds:.2f}s")

    # -----------------------------------------
    # Release large fold objects
    # -----------------------------------------

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.032902
PR lift   : 33.72x
ROC-AUC   : 0.887220
Recall    : 80.0425%
Precision : 0.5674%
FP        : 66,063
Alerts    : 66,440
Alert rate: 13.7657%
Fit time  : 16.65s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.047368
PR lift   : 44.46x
ROC-AUC   : 0.917476
Recall    : 80.0584%
Precision : 1.5732%
FP        : 51,490
Alerts    : 52,313
Alert rate: 5.4219%
Fit time  : 19.61s


In [45]:
lightgbm_results = build_fold_metrics_table(fold_results)

lightgbm_results[
    [
        "fold",
        "rows",
        "frauds",
        "prevalence",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,rows,frauds,prevalence,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,207430,407,0.001962,0.063079,32.148566,0.941933,0.022770,0.800983,13991,14317,0.069021
1,fold_2,482650,471,0.000976,0.032902,33.716026,0.887220,0.005674,0.800425,66063,66440,0.137657
2,fold_3,964840,1028,0.001065,0.047368,44.457524,0.917476,0.015732,0.800584,51490,52313,0.054219


In [46]:
lightgbm_stability = summarize_temporal_stability(lightgbm_results)

lightgbm_stability.T

,0
mean_average_precision,0.047783
std_average_precision,0.012323
min_average_precision,0.032902
max_average_precision,0.063079
range_average_precision,0.030177
latest_fold_average_precision,0.047368
mean_pr_lift,36.774039
latest_fold_pr_lift,44.457524
mean_precision_at_recall_floor,0.014726
mean_alert_rate_at_recall_floor,0.086966


In [47]:
fold_3_ap = lightgbm_results.loc[
    lightgbm_results["fold"] == "fold_3",
    "average_precision",
].iloc[0]

SGD_BASELINE_AP = 0.010280

print(f"SGD Baseline AP   : {SGD_BASELINE_AP:.6f}")
print(f"LightGBM Fold 3 AP: {fold_3_ap:.6f}")
print(f"Absolute change   : {fold_3_ap - SGD_BASELINE_AP:+.6f}")
print(f"Ratio vs SGD      : {fold_3_ap / SGD_BASELINE_AP:.2f}x")

SGD Baseline AP   : 0.010280
LightGBM Fold 3 AP: 0.047368
Absolute change   : +0.037088
Ratio vs SGD      : 4.61x


# CatBGM

In [14]:
dev_df = dev_df.sort_values(
    "transaction_timestamp",
    kind="stable",
).reset_index(drop=True)

## Inspect missing values

In [15]:
missing = [
    col
    for col in (FEATURE_COLUMNS + [TARGET_COL, TIMESTAMP_COL])
    if col not in dev_df.columns
]

missing

[]

In [16]:
dev_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].isna().sum()

amount_received        0
amount_paid            0
hour_of_day            0
day_of_week            0
is_weekend             0
same_currency_flag     0
same_bank_flag         0
log_amount_received    0
log_amount_paid        0
receiving_currency     0
payment_currency       0
payment_format         0
dtype: int64

In [17]:
dev_df[CATEGORICAL_FEATURES].dtypes

receiving_currency    str
payment_currency      str
payment_format        str
dtype: object

In [18]:
def prepare_catboost_features(
    df: pd.DataFrame,
) -> pd.DataFrame:
    X = df[FEATURE_COLUMNS].copy()

    for col in CATEGORICAL_FEATURES:
        X[col] = (
            X[col]
            .where(
                X[col].notna(),
                "__MISSING__",
            )
            .astype(str)
        )

    return X

## Recreate and verify frozen splits

In [19]:
splits = build_walkforward_splits(dev_df)


In [20]:
for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    y_train = dev_df.iloc[train_idx][TARGET_COL]

    y_val = dev_df.iloc[val_idx][TARGET_COL]

    print(
        fold.name,
        "| train:",
        f"{len(train_idx):,}",
        f"fraud={y_train.sum():,}",
        "| val:",
        f"{len(val_idx):,}",
        f"fraud={y_val.sum():,}",
    )

fold_1 | train: 2,076,752 fraud=1,121 | val: 207,430 fraud=407
fold_2 | train: 2,284,182 fraud=1,528 | val: 482,650 fraud=471
fold_3 | train: 2,766,832 fraud=1,999 | val: 964,840 fraud=1,028


## Fold 1 only

In [21]:
fold = FROZEN_FOLDS[0]

train_idx, val_idx = splits[0]

train_df = dev_df.iloc[train_idx].sort_values(
    TIMESTAMP_COL,
    kind="stable",
)

val_df = dev_df.iloc[val_idx].sort_values(
    TIMESTAMP_COL,
    kind="stable",
)

In [22]:
X_train = prepare_catboost_features(train_df)


In [23]:
y_train = train_df[TARGET_COL].to_numpy()


In [24]:
X_val = prepare_catboost_features(val_df)


In [25]:
y_val = val_df[TARGET_COL].to_numpy()

In [26]:
print(
    "Train:",
    X_train.shape,
    "| Fraud:",
    y_train.sum(),
)

print(
    "Validation:",
    X_val.shape,
    "| Fraud:",
    y_val.sum(),
)

Train: (2076752, 12) | Fraud: 1121
Validation: (207430, 12) | Fraud: 407


In [27]:
train_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train,
)

In [28]:
positive_weight = train_weights[y_train == 1][0]

negative_weight = train_weights[y_train == 0][0]

print(f"Positive weight: {positive_weight:.4f}")

print(f"Negative weight: {negative_weight:.4f}")

print(
    "Ratio:",
    positive_weight / negative_weight,
)

Positive weight: 926.2944
Negative weight: 0.5003
Ratio: 1851.5887600356823


## CatBoost Pools

In [29]:
train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=CATEGORICAL_FEATURES,
    weight=train_weights,
)

val_pool = Pool(
    data=X_val,
    label=y_val,
    cat_features=CATEGORICAL_FEATURES,
)

## CatBoost configuration

In [30]:
catboost_model = CatBoostClassifier(
    loss_function="Logloss",
    iterations=300,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3.0,
    random_seed=42,
    # Important for our chronological data
    has_time=True,
    # Be explicit for this large CPU experiment
    boosting_type="Plain",
    thread_count=-1,
    # Don't create catboost_info/ on disk
    allow_writing_files=False,
    verbose=False,
)

## Train Fold 1

In [31]:
start = perf_counter()

catboost_model.fit(train_pool)

fit_seconds = perf_counter() - start

print(f"CatBoost Fold 1 fit time: {fit_seconds:.2f} seconds")

CatBoost Fold 1 fit time: 100.88 seconds


## validation scores - fold 1

In [32]:
start = perf_counter()

catboost_val_scores = catboost_model.predict_proba(val_pool)[:, 1]

predict_seconds = perf_counter() - start

print(f"Prediction time: {predict_seconds:.2f} seconds")

Prediction time: 0.04 seconds


## Fold 1 - Inspect the score distribution

In [33]:
pd.Series(catboost_val_scores).describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.995,
    ]
)

count    2.074300e+05
mean     2.485987e-01
std      2.788105e-01
min      4.720994e-07
50%      1.617559e-01
75%      3.572339e-01
90%      6.842705e-01
95%      9.602768e-01
99%      9.764094e-01
99.5%    9.790713e-01
max      9.853335e-01
dtype: float64

In [34]:
catboost_fold_1_metrics = evaluate_fold_scores(
    fold_name="fold_1",
    y_true=y_val,
    y_score=catboost_val_scores,
    min_recall=0.80,
)

catboost_fold_1_metrics

{'fold': 'fold_1',
 'rows': 207430,
 'frauds': 407,
 'prevalence': 0.0019621076989827894,
 'average_precision': 0.05646067595623898,
 'pr_lift': 28.775523374945088,
 'roc_auc': 0.941029727601751,
 'recall_floor': 0.8,
 'diagnostic_cutoff': 0.9329158520132423,
 'precision_at_recall_floor': 0.022676683361157485,
 'recall_at_recall_floor': 0.800982800982801,
 'false_positives_at_recall_floor': 14050,
 'true_positives_at_recall_floor': 326,
 'false_negatives_at_recall_floor': 81,
 'true_negatives_at_recall_floor': 192973,
 'alerts_at_recall_floor': 14376,
 'alert_rate_at_recall_floor': 0.06930530781468447}

In [35]:
print(f"Average Precision: {catboost_fold_1_metrics['average_precision']:.6f}")

print(f"PR lift: {catboost_fold_1_metrics['pr_lift']:.2f}x")

print(f"ROC-AUC: {catboost_fold_1_metrics['roc_auc']:.6f}")

print(f"Recall: {catboost_fold_1_metrics['recall_at_recall_floor']:.4%}")

print(
    f"Precision @ >=80% recall: "
    f"{catboost_fold_1_metrics['precision_at_recall_floor']:.4%}"
)

print(
    f"False positives: {catboost_fold_1_metrics['false_positives_at_recall_floor']:,}"
)

print(f"Alerts: {catboost_fold_1_metrics['alerts_at_recall_floor']:,}")

print(f"Alert rate: {catboost_fold_1_metrics['alert_rate_at_recall_floor']:.4%}")

print(f"Fit time: {fit_seconds:.2f}s")

Average Precision: 0.056461
PR lift: 28.78x
ROC-AUC: 0.941030
Recall: 80.0983%
Precision @ >=80% recall: 2.2677%
False positives: 14,050
Alerts: 14,376
Alert rate: 6.9305%
Fit time: 100.88s


## Run 2 & 3

In [36]:
catboost_fold_results = [catboost_fold_1_metrics]

catboost_runtime_results = []

In [37]:
for fold_number in [1, 2]:
    fold = FROZEN_FOLDS[fold_number]
    train_idx, val_idx = splits[fold_number]

    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    # -----------------------------------------
    # Chronological fold data
    # -----------------------------------------

    train_df = dev_df.iloc[train_idx].sort_values(
        TIMESTAMP_COL,
        kind="stable",
    )

    val_df = dev_df.iloc[val_idx].sort_values(
        TIMESTAMP_COL,
        kind="stable",
    )

    X_train = prepare_catboost_features(train_df)

    y_train = train_df[TARGET_COL].to_numpy()

    X_val = prepare_catboost_features(val_df)

    y_val = val_df[TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    # -----------------------------------------
    # Same balanced-weight strategy
    # -----------------------------------------

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    # -----------------------------------------
    # CatBoost Pool
    # -----------------------------------------

    train_pool = Pool(
        data=X_train,
        label=y_train,
        cat_features=CATEGORICAL_FEATURES,
        weight=train_weights,
    )

    val_pool = Pool(
        data=X_val,
        label=y_val,
        cat_features=CATEGORICAL_FEATURES,
    )

    # -----------------------------------------
    # Fresh CatBoost model for every fold
    # -----------------------------------------

    model = CatBoostClassifier(
        loss_function="Logloss",
        iterations=300,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=42,
        has_time=True,
        boosting_type="Plain",
        thread_count=-1,
        allow_writing_files=False,
        verbose=False,
    )

    # -----------------------------------------
    # Fit
    # -----------------------------------------

    start = perf_counter()

    model.fit(train_pool)

    fit_seconds = perf_counter() - start

    # -----------------------------------------
    # Predict
    # -----------------------------------------

    start = perf_counter()

    val_scores = model.predict_proba(val_pool)[:, 1]

    predict_seconds = perf_counter() - start

    # -----------------------------------------
    # Same common evaluator
    # -----------------------------------------

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    catboost_fold_results.append(metrics)

    catboost_runtime_results.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")

    print(f"PR lift   : {metrics['pr_lift']:.2f}x")

    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")

    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")

    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")

    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")

    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")

    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")

    print(f"Fit time  : {fit_seconds:.2f}s")

    # -----------------------------------------
    # Release fold memory
    # -----------------------------------------

    del (
        model,
        train_pool,
        val_pool,
        train_df,
        val_df,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.023961
PR lift   : 24.55x
ROC-AUC   : 0.893627
Recall    : 80.0425%
Precision : 1.1281%
FP        : 33,043
Alerts    : 33,420
Alert rate: 6.9243%
Fit time  : 101.97s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.034632
PR lift   : 32.50x
ROC-AUC   : 0.914625
Recall    : 80.0584%
Precision : 1.6480%
FP        : 49,116
Alerts    : 49,939
Alert rate: 5.1759%
Fit time  : 119.41s


In [38]:
catboost_results = build_fold_metrics_table(catboost_fold_results)

catboost_results[
    [
        "fold",
        "rows",
        "frauds",
        "prevalence",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,rows,frauds,prevalence,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,207430,407,0.001962,0.056461,28.775523,0.941030,0.022677,0.800983,14050,14376,0.069305
1,fold_2,482650,471,0.000976,0.023961,24.553352,0.893627,0.011281,0.800425,33043,33420,0.069243
2,fold_3,964840,1028,0.001065,0.034632,32.504135,0.914625,0.016480,0.800584,49116,49939,0.051759


In [39]:
catboost_stability = summarize_temporal_stability(catboost_results)

catboost_stability.T

,0
mean_average_precision,0.038351
std_average_precision,0.013526
min_average_precision,0.023961
max_average_precision,0.056461
range_average_precision,0.032500
latest_fold_average_precision,0.034632
mean_pr_lift,28.611003
latest_fold_pr_lift,32.504135
mean_precision_at_recall_floor,0.016812
mean_alert_rate_at_recall_floor,0.063436
